<a href="https://colab.research.google.com/github/giordanoandrade-glitch/Meu-site/blob/main/MVSepLess_Dzeta_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Установка (1 минута)
!git clone https://github.com/noblebarkrr/mvsepless -b dzeta
!pip install uv
!uv pip install --no-cache-dir -r mvsepless/requirements.txt
!python mvsepless/inference.py info -update

Cloning into 'mvsepless'...
remote: Enumerating objects: 3083, done.
remote: Counting objects: 100% (322/322), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 3083 (delta 211), reused 100 (delta 67), pack-reused 2761 (from 2)
Receiving objects: 100% (3083/3083), 34.93 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (1819/1819), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 66.0 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 179 packages in 3.38s
Prepared 45 packages in 39.27s
Uninstalled 1 package in 0.90ms
Installed 45 packages in 301ms
 + asteroid==0.7.0
 + asteroid-filterbanks==0.4.0
 + audiomentations==0.43.1
 + bitsandbytes==0.49.2
 + cached-property==2.0.1
 + diffq==0.2.4
 + faiss-cpu==1.11.0
 + hyper-connections==0.4.11
 + julius==0.2.7
 + lightning-utilities==0.15.3
 + local-attention==1.11.2
 + mir-eval==0.8.2
 + ml-collections==1.1.0
 + numpy-minmax==0.5.0
 + numpy-rms==0.6.0
 + onnx==1.21.0
 + onnx2torch==1.5.15
 + 

In [2]:
#@title Привязать Google Диск
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from pyngrok import ngrok
import random
import string
import re
import urllib
import time
import ipywidgets as widgets
from IPython.display import display, Javascript
import threading
import subprocess

#@title # Web-UI
port = 7934
sharing_method = "gradio" # @param ["gradio","ngrok","localtunnel","not"]
#@markdown ###### Токен для ngrok *(где взять его - https://dashboard.ngrok.com/get-started/your-authtoken)*
ngrok_token = "" # @param {"type":"string","placeholder":"токен"}

lt_sub_domain = "mvsepless"
def generate_subdomain(length=8):
    """Генерация случайного субдомена заданной длины"""
    chars = string.ascii_lowercase + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

if sharing_method == "ngrok":
    try:
        ngrok.set_auth_token(ngrok_token)
        ngrok.kill()
        tunnel = ngrok.connect(port)
        print(f"Публичная ссылка: {tunnel.public_url}")
    except KeyboardInterrupt:
        ngrok.kill()

if sharing_method == "localtunnel":
    os.system("npm install -g localtunnel &>/dev/null")
    time.sleep(7)
    with open('url.txt', 'w') as file:
        file.write('')
    subdomain = f"{re.sub(r'[^a-zA-Z0-9]', '', lt_sub_domain)}-{generate_subdomain(25)}"

    # Флаг для контроля работы потока
    tunnel_running = True

    def run_tunnel():
        while tunnel_running:
            print("localtunnel включается...")
            try:
                # Используем subprocess вместо os.system для лучшего контроля
                process = subprocess.Popen(
                    f'lt --port {port} '
                    f'{f"--subdomain {subdomain}" if lt_sub_domain != "" and not lt_sub_domain.isspace() else ""}',
                    shell=True,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE
                )
                process.wait()  # Ждем завершения процесса
                if not tunnel_running:
                    break
                time.sleep(5)  # Пауза перед перезапуском
            except Exception as e:
                if tunnel_running:
                    print(f"Ошибка в localtunnel: {e}")
                    time.sleep(5)

    tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
    tunnel_thread.start()

    time.sleep(3)
    try:
        endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        tunnel_url = f"https://{subdomain}.loca.lt"
        print(f"Публичная ссылка: {tunnel_url}")

        # Создаем текстовое поле с URL, а не IP
        text_field = widgets.Text(
            value=endpoint_ip,  # Исправлено: показываем URL, а не IP
            description='URL:',
            disabled=True
        )
        text_field.add_class("copy-enabled")

        display(text_field)

        # Исправленный JavaScript для копирования
        display(Javascript("""
        setTimeout(() => {
            const input = document.querySelector('.copy-enabled input');
            if (!input) return;

            const btn = document.createElement('button');
            btn.innerHTML = '📋';
            btn.style.cssText = `
                margin-left: 8px;
                border: none;
                background: none;
                cursor: pointer;
                font-size: 1.2em;
            `;
            input.parentNode.appendChild(btn);

            btn.addEventListener('click', () => {
                navigator.clipboard.writeText(input.value)  // Исправлено: input.value вместо input
                    .then(() => {
                        btn.innerHTML = '✓';
                        setTimeout(() => btn.innerHTML = '📋', 2000);
                    })
                    .catch(err => {
                        console.error('Ошибка копирования: ', err);
                    });
            });
        }, 300);
        """))

    except Exception as e:
        print(f"Ошибка при старте localtunnel: {e}")

    # Функция для корректного завершения
    def stop_tunnel():
        global tunnel_running
        tunnel_running = False
        print("Localtunnel завершает работу...")

    # Регистрируем обработчик для Ctrl+C
    import signal
    original_signal_handler = signal.getsignal(signal.SIGINT)

    def signal_handler(sig, frame):
        stop_tunnel()
        # Восстанавливаем оригинальный обработчик и вызываем его
        signal.signal(signal.SIGINT, original_signal_handler)
        raise KeyboardInterrupt

    signal.signal(signal.SIGINT, signal_handler)

cmd = ["python", "/content/mvsepless/app.py", "--port", str(port), "--full"]
if sharing_method == "gradio":
    cmd.append("--share")

!{' '.join(cmd)}

[Статус] Начало загрузки, размер: 172.79 МБ
rmvpe.pt: 100% 173M/173M [00:01<00:00, 171MB/s]
[Статус] ✓ Загрузка завершена: /content/mvsepless/vbach_lib/predictors/rmvpe.pt
[Статус] Начало загрузки, размер: 64.80 МБ
hpa_rmvpe.pt: 100% 64.8M/64.8M [00:00<00:00, 166MB/s]
[Статус] ✓ Загрузка завершена: /content/mvsepless/vbach_lib/predictors/hpa_rmvpe.pt
[Статус] Начало загрузки, размер: 65.81 МБ
fcpe.pt: 100% 65.8M/65.8M [00:00<00:00, 197MB/s]
[Статус] ✓ Загрузка завершена: /content/mvsepless/vbach_lib/predictors/fcpe.pt


# CLI

## Разделение

### Инференс

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
model_name = "bs_6stem" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
template = "NAME_STEM_MODEL" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]

cmd = [
    "python", "mvsepless/inference.py", "separate",
    "-i", shlex.quote(input_file),
    "-o", shlex.quote(output_dir),
    "-of", shlex.quote(output_format),
    "-tm", shlex.quote(template),
    "-mn", shlex.quote(model_name)
]

quoted_string = " ".join(cmd)
!{quoted_string}

### Инференс с кастомной моделью

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
checkpoint_path = "" # @param {"type":"string","placeholder":"путь/к/модели.pth"}
config_path = "" # @param {"type":"string","placeholder":"путь/к/конфигу.yaml"}
model_type = "bs_roformer" # @param ["mel_band_roformer","bs_roformer","mdx23c","scnet","scnet_masked","scnet_tran","htdemucs","bandit","bandit_v2","mdxnet","vr","medley_vox"]
template = "NAME_STEM_MODEL" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]

cmd = [
    "python", "mvsepless/inference.py", "custom_separate",
    "-i", shlex.quote(input_file),
    "-o", shlex.quote(output_dir),
    "-of", shlex.quote(output_format),
    "-tm", shlex.quote(template),
    "-mt", shlex.quote(model_type),
    "-ckpt", shlex.quote(checkpoint_path),
    "-conf", shlex.quote(config_path)
]

quoted_string = " ".join(cmd)
!{quoted_string}

### Информация о моделях

In [ ]:
import shlex

limit = 0 # @param {"type":"number","placeholder":"лимит отображаемых моделей"}
stem = "" # @param {"type":"string","placeholder":"фильтр по стему"}
only_installed = False # @param {"type":"boolean"}


cmd = ["python", "mvsepless/inference.py", "info"]
if limit:
    cmd.extend(["-l", str(limit)])
if stem:
    cmd.extend(["-s", shlex.quote(stem)])
if only_installed:
    cmd.extend(["-oi"])

quoted_string = " ".join(cmd)
!{quoted_string}

### Скачать модель

In [ ]:
#@title Модель из списка
import shlex

model_name = "bs_6stem" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}

cmd = [
    "python", "mvsepless/inference.py", "info", "--download",
    "-model", shlex.quote(model_name)
]

quoted_string = " ".join(cmd)
!{quoted_string}

In [ ]:
#@title Кастомная модель
import sys
from urllib.parse import urlparse
from pathlib import Path, PurePosixPath
sys.path.append(str(Path.cwd() / "mvsepless"))
from namer import Namer
from extra_utils import dw_file
custom_cache_dir = Path("custom_cache")
custom_cache_dir.mkdir(exist_ok=True, parents=True)
checkpoint_url = "" # @param {"type":"string","placeholder":"https;//путь/к/модели.pth"}
config_url = "" # @param {"type":"string","placeholder":"https://путь/к/конфигу.yaml"}

if checkpoint_url:
    clean_ckpt_url = urlparse(checkpoint_url)._replace(query="", fragment="").geturl()
    ckpt_file_name = PurePosixPath(PurePosixPath(clean_ckpt_url).name)
    if ckpt_file_name.suffix != ".ckpt":
        ckpt_file_name = ckpt_file_name.with_suffix(".ckpt")
    dw_file(checkpoint_url, Namer.iter(custom_cache_dir / ckpt_file_name))

if config_url:
    clean_conf_url = urlparse(config_url)._replace(query="", fragment="").geturl()
    conf_file_name = PurePosixPath(PurePosixPath(clean_conf_url).name)
    if conf_file_name.suffix != ".yaml":
        conf_file_name = conf_file_name.with_suffix(".yaml")
    dw_file(checkpoint_url, Namer.iter(custom_cache_dir / conf_file_name))



### Очистить кэш

In [ ]:
#@title Это действие не обратимо
!python mvsepless/inference.py info -clear

## Ансамбль

#### Авто-ансамбль (до 10 моделей)

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
template = "NAME_TYPE_COUNT" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]
ensemble_type = "max_fft" # @param ["avg_fft","median_fft","min_fft","max_fft"]
save_primary_stems = False # @param {type:"boolean"}
#@markdown ---
#@markdown #### Пресет
#@markdown ---
model_name_1 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_1 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_1 = False # @param {type:"boolean"}
weight_1 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_2 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_2 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_2 = False # @param {type:"boolean"}
weight_2 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_3 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_3 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_3 = False # @param {type:"boolean"}
weight_3 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_4 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_4 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_4 = False # @param {type:"boolean"}
weight_4 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_5 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_5 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_5 = False # @param {type:"boolean"}
weight_5 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_6 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_6 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_6 = False # @param {type:"boolean"}
weight_6 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_7 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_7 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_7 = False # @param {type:"boolean"}
weight_7 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_8 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_8 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_8 = False # @param {type:"boolean"}
weight_8 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_9 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_9 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_9 = False # @param {type:"boolean"}
weight_9 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
model_name_10 = "" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_bigbeta7_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_karaoke_small_gabox_aufr33', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_denoise_yuluoye', 'mbr_denoise_children_phaedrus33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_hybrid_arch_aname', 'mbr_scratch_aname', 'mbr_bgm_jasper', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_inst_rifforge_meskvlla33', 'mbr_neo_inst_vfx', 'mbr_lead_rhythm_guitar_listra92', 'mbr_guitar_chencfd', 'mbr_mid_side_gilliaaan', 'mbr_vocals_zfturbo', 'mbr_amb_jazzpear', 'mbr_expl_jazzpear', 'mbr_fight_jazzpear', 'mbr_foot_jazzpear', 'mbr_gen_jazzpear', 'mbr_misc_jazzpear', 'mbr_toon_jazzpear', 'mbr_speech_alicen', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_vocals_1297_viperx', 'bs_other_viperx', 'bs_siamese_vocals_unwa', 'bs_inst_exp_vlp_unwa', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_vocals_large1_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_resurrection_inst_gabox', 'bs_inst_fno_unwa', 'bs_inst_large2_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_karaoke_inv_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_vocalsft1_anvuew', 'bs_mag_anvuew', 'bs_4stem_aname', 'bs_karaoke_3stem_giantailab', 'bs_vocals1_aname', 'bs_vocals2_aname', 'bs_orch_xlancer', 'bs_orch2_xlancer', 'bs_keys_xlancer', 'bs_bass_xlancer', 'bs_drums_xlancer', 'bs_drums2_xlancer', 'bs_gtr_xlancer', 'bs_perc_xlancer', 'bs_perc2_xlancer', 'bs_syn_xlancer', 'bs_syn2_xlancer', 'bs_vox_xlancer', 'bs_drums_gilliaaan', 'bs_mid_side1_gilliaaan', 'bs_mid_side2_gilliaaan', 'bs_speech_alicen', 'bs_pope_4stem_aname', 'bs_pope_instvoc_aname', 'bs_pope_vocals_zfturbo', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_mid_side_gilliaaan', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx23c_sfx_jasper', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_huge_4stem1.2_aname', 'scnet_huge_4stem_fullness_aname', 'scnet_huge_4stem_str_fullness_aname', 'scnet_huge_4stem_bleedless_aname', 'scnet_mid_side_gilliaaan', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'scnet_masked_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'demucs4_drumsep_4stem_inagoy', 'demucs4_cdx_zfturbo_1', 'demucs4_cdx_zfturbo_2', 'demucs4_cdx_zfturbo_3', 'demucs3_saxophone', 'bandit_plus', 'bandit_30_zfturbo', 'bandit_57_zfturbo', 'bandit_63_zfturbo', 'bandit_last', 'bandit_v2_multi', 'multi_singing_librispeech', 'multi_singing_librispeech_138', 'singing_librispeech_ft_isrnet', 'singing_librispeech_isrnet', 'medley_vox_vocal_231', 'medley_vox_vocals_135', 'medley_vox_vocals_163', 'medley_vox_vocals_188', 'medley_vox_vocals_200', 'medley_vox_vocals_238'] {"allow-input":true}
primary_stem_10 = "" # @param {"type":"string","placeholder":"основной стем"}
invert_10 = False # @param {type:"boolean"}
weight_10 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---

cmd = [
    "python", "mvsepless/inference.py", "auto_ensemble",
    "-i", shlex.quote(input_file),
    "-o", shlex.quote(output_dir),
    "-of", shlex.quote(output_format),
    "-tm", shlex.quote(template),
    "-t", shlex.quote(ensemble_type)
]
if save_primary_stems:
  cmd.append("-save_stems")
flow_cmd = [
    "-flow"
]

if model_name_1:
  flow_cmd.append(shlex.quote(f"{model_name_1}:{primary_stem_1}:{invert_1}:{weight_1}"))
if model_name_2:
  flow_cmd.append(shlex.quote(f"{model_name_2}:{primary_stem_2}:{invert_2}:{weight_2}"))
if model_name_3:
  flow_cmd.append(shlex.quote(f"{model_name_3}:{primary_stem_3}:{invert_3}:{weight_3}"))
if model_name_4:
  flow_cmd.append(shlex.quote(f"{model_name_4}:{primary_stem_4}:{invert_4}:{weight_4}"))
if model_name_5:
  flow_cmd.append(shlex.quote(f"{model_name_5}:{primary_stem_5}:{invert_5}:{weight_5}"))
if model_name_6:
  flow_cmd.append(shlex.quote(f"{model_name_6}:{primary_stem_6}:{invert_6}:{weight_6}"))
if model_name_7:
  flow_cmd.append(shlex.quote(f"{model_name_7}:{primary_stem_7}:{invert_7}:{weight_7}"))
if model_name_8:
  flow_cmd.append(shlex.quote(f"{model_name_8}:{primary_stem_8}:{invert_8}:{weight_8}"))
if model_name_9:
  flow_cmd.append(shlex.quote(f"{model_name_9}:{primary_stem_9}:{invert_9}:{weight_9}"))
if model_name_10:
  flow_cmd.append(shlex.quote(f"{model_name_10}:{primary_stem_10}:{invert_10}:{weight_10}"))

cmd.extend(flow_cmd)
quoted_string = " ".join(cmd)
!{quoted_string}

#### Ручной ансамбль (до 10 файлов)

In [ ]:
import shlex

output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
template = "NAME_TYPE" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]
ensemble_type = "max_fft" # @param ["avg_fft","median_fft","min_fft","max_fft"]
#@markdown ---
#@markdown #### Входные файлы
#@markdown ---
input_file_1 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_1 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_2 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_2 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_3 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_3 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_4 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_4 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_5 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_5 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_6 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_6 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_7 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_7 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_8 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_8 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_9 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_9 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---
input_file_10 = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
weight_10 = 1 # @param {"type":"number","placeholder":"веса"}
#@markdown ---

cmd = [
    "python", "mvsepless/inference.py", "manual_ensemble",
    "-o", shlex.quote(output_dir),
    "-of", shlex.quote(output_format),
    "-tm", shlex.quote(template),
    "-t", shlex.quote(ensemble_type),
]
files_cmd = ["-i"]
weights_cmd = ["-weights"]
if input_file_1:
  files_cmd.append(shlex.quote(input_file_1))
  weights_cmd.append(str(weight_1))
if input_file_2:
  files_cmd.append(shlex.quote(input_file_2))
  weights_cmd.append(str(weight_2))
if input_file_3:
  files_cmd.append(shlex.quote(input_file_3))
  weights_cmd.append(str(weight_3))
if input_file_4:
  files_cmd.append(shlex.quote(input_file_4))
  weights_cmd.append(str(weight_4))
if input_file_5:
  files_cmd.append(shlex.quote(input_file_5))
  weights_cmd.append(str(weight_5))
if input_file_6:
  files_cmd.append(shlex.quote(input_file_6))
  weights_cmd.append(str(weight_6))
if input_file_7:
  files_cmd.append(shlex.quote(input_file_7))
  weights_cmd.append(str(weight_7))
if input_file_8:
  files_cmd.append(shlex.quote(input_file_8))
  weights_cmd.append(str(weight_8))
if input_file_9:
  files_cmd.append(shlex.quote(input_file_9))
  weights_cmd.append(str(weight_9))
if input_file_10:
  files_cmd.append(shlex.quote(input_file_10))
  weights_cmd.append(str(weight_10))

cmd.extend(files_cmd)
cmd.extend(weights_cmd)

quoted_string = " ".join(cmd)
!{quoted_string}

## Преобразование

### Инференс

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
model_path = "" # @param {"type":"string","placeholder":"путь/к/модели.pth"}
index_path = "" # @param {"type":"string","placeholder":"путь/к/индекс-файлу.index"}
pitch = 0 # @param {type:"number"}
f0_method = "rmvpe+" # @param ["rmvpe+","hpa-rmvpe","fcpe","mangio-crepe","mangio-crepe-tiny","harvest","pm","pyin"]
index_rate = 0 # @param {"type":"slider","min":0,"max":1,"step":0.01}
volume_envelope = 0 # @param {type:"number"}
protect = 0.33 # @param {"type":"slider","min":0,"max":0.5,"step":0.01}
hop_length = 128 # @param {"type":"slider","min":32,"max":512,"step":8}
template = "NAME_F0METHOD_PITCH" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]
stereo_mode = "mono" # @param ["mono","left/right","sim/dif"]
embedder = "fairseq | hubert_base" # @param ["fairseq | hubert_base","fairseq | contentvec_base","fairseq | korean_hubert_base","fairseq | chinese_hubert_base","fairseq | portuguese_hubert_base","fairseq | japanese_hubert_base","transformers | contentvec","transformers | spin","transformers | spin-v2","transformers | chinese-hubert-base","transformers | japanese-hubert-base","transformers | korean-hubert-base"]
f0_min = 50 # @param {"type":"integer","placeholder":"минимальная частота f0"}
f0_max = 1100 # @param {"type":"integer","placeholder":"максимальная частота f0"}
chunk_duration = 7 # @param {"type":"integer","placeholder":"время в сек"}


cmd = [
    "python", "mvsepless/vbach_lib/infer.py", "infer",
    "--input", shlex.quote(input_file),
    "--output", shlex.quote(output_dir),
    "--model_path", shlex.quote(model_path),
    "--index_path", shlex.quote(index_path),
    "--pitch", str(pitch),
    "-f0m", shlex.quote(f0_method),
    "-idxr", str(index_rate),
    "-ve", str(volume_envelope),
    "-pr", str(protect),
    "-hl", str(hop_length),
    "-of", shlex.quote(output_format),
    "-stm", shlex.quote(stereo_mode),
    "-f0min", str(f0_min),
    "-f0max", str(f0_max),
    "-chd", str(chunk_duration),
    "-tm", shlex.quote(template)
]

if embedder.startswith("fairseq"):
    cmd.extend([
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])
elif embedder.startswith("transformers"):
    cmd.extend([
        "-tf",
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])

quoted_string = " ".join(cmd)
!{quoted_string}

### Инференс c кастомным F0

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_dir = "" # @param {"type":"string","placeholder":"путь/к/директории"}
model_path = "" # @param {"type":"string","placeholder":"путь/к/модели.pth"}
index_path = "" # @param {"type":"string","placeholder":"путь/к/индекс-файлу.index"}
pitch = 0 # @param {type:"number"}
f0_file = "" # @param {"type":"string","placeholder":"путь/к/кривой_f0.json"}
index_rate = 0 # @param {"type":"slider","min":0,"max":1,"step":0.01}
volume_envelope = 0 # @param {type:"number"}
protect = 0.33 # @param {"type":"slider","min":0,"max":0.5,"step":0.01}
template = "NAME_F0METHOD_PITCH" # @param {"type":"string"}
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "ac3", "aiff", "wma"]
stereo_mode = "mono" # @param ["mono","left/right","sim/dif"]
embedder = "fairseq | hubert_base" # @param ["fairseq | hubert_base","fairseq | contentvec_base","fairseq | korean_hubert_base","fairseq | chinese_hubert_base","fairseq | portuguese_hubert_base","fairseq | japanese_hubert_base","transformers | contentvec","transformers | spin","transformers | spin-v2","transformers | chinese-hubert-base","transformers | japanese-hubert-base","transformers | korean-hubert-base"]
f0_min = 50 # @param {"type":"integer","placeholder":"минимальная частота f0"}
f0_max = 1100 # @param {"type":"integer","placeholder":"максимальная частота f0"}
chunk_duration = 7 # @param {"type":"integer","placeholder":"время в сек"}


cmd = [
    "python", "mvsepless/vbach_lib/infer.py", "infer_custom_f0",
    "-input", shlex.quote(input_file),
    "-output", shlex.quote(output_dir),
    "--model_path", shlex.quote(model_path),
    "--index_path", shlex.quote(index_path),
    "--pitch", str(pitch),
    "-f0f", shlex.quote(f0_file),
    "-idxr", str(index_rate),
    "-ve", str(volume_envelope),
    "-pr", str(protect),
    "-of", shlex.quote(output_format),
    "-f0min", str(f0_min),
    "-f0max", str(f0_max),
    "-chd", str(chunk_duration),
    "-tm", shlex.quote(template)
]

if embedder.startswith("fairseq"):
    cmd.extend([
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])
elif embedder.startswith("transformers"):
    cmd.extend([
        "-tf",
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])

quoted_string = " ".join(cmd)
!{quoted_string}

### Извлечь F0

In [ ]:
import shlex

input_file = "" # @param {"type":"string","placeholder":"путь/к/файлу.mp3"}
output_path = "" # @param {"type":"string","placeholder":"путь/к/файлу.json"}
f0_method = "rmvpe+" # @param ["rmvpe+","hpa-rmvpe","fcpe","mangio-crepe","mangio-crepe-tiny","harvest","pm","pyin"]
f0_min = 50 # @param {"type":"integer","placeholder":"минимальная частота f0"}
f0_max = 1100 # @param {"type":"integer","placeholder":"максимальная частота f0"}


cmd = [
    "python", "mvsepless/vbach_lib/f0_extractor.py",
    "-input", shlex.quote(input_file),
    "-f0m", shlex.quote(f0_method),
    "-f0min", str(f0_min),
    "-f0max", str(f0_max),
    "-output", shlex.quote(output_path)
]

quoted_string = " ".join(cmd)
!{quoted_string}

### Скачать эмбеддер

In [ ]:
import shlex

cmd = [
    "python", "mvsepless/vbach_lib/infer.py", "download_hubert",
]
embedder = "fairseq | hubert_base" # @param ["fairseq | hubert_base","fairseq | contentvec_base","fairseq | korean_hubert_base","fairseq | chinese_hubert_base","fairseq | portuguese_hubert_base","fairseq | japanese_hubert_base","transformers | contentvec","transformers | spin","transformers | spin-v2","transformers | chinese-hubert-base","transformers | japanese-hubert-base","transformers | korean-hubert-base"]

if embedder.startswith("fairseq"):
    cmd.extend([
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])
elif embedder.startswith("transformers"):
    cmd.extend([
        "-tf",
        "-emb", shlex.quote(embedder.split(" | ")[1])
    ])

quoted_string = " ".join(cmd)
!{quoted_string}

### Скачать модель

In [ ]:
import sys
from urllib.parse import urlparse
from pathlib import Path, PurePosixPath
sys.path.append(str(Path.cwd() / "mvsepless"))
from namer import Namer
from extra_utils import dw_file
import tempfile
import zipfile
custom_cache_dir = Path("custom_voices")
custom_cache_dir.mkdir(exist_ok=True, parents=True)
checkpoint_url = "" # @param {"type":"string","placeholder":"https;//путь/к/модели.pth"}
index_url = "" # @param {"type":"string","placeholder":"https://путь/к/конфигу.yaml"}
zip_url = "" # @param {"type":"string","placeholder":"https://путь/к/архиву.zip"}

if checkpoint_url:
    clean_ckpt_url = urlparse(checkpoint_url)._replace(query="", fragment="").geturl()
    ckpt_file_name = PurePosixPath(PurePosixPath(clean_ckpt_url).name)
    if ckpt_file_name.suffix != ".ckpt":
        ckpt_file_name = ckpt_file_name.with_suffix(".ckpt")
    dw_file(checkpoint_url, Namer.iter(custom_cache_dir / ckpt_file_name))

if index_url:
    clean_index_url = urlparse(index_url)._replace(query="", fragment="").geturl()
    index_file_name = PurePosixPath(PurePosixPath(clean_index_url).name)
    if index_file_name.suffix != ".index":
        index_file_name = index_file_name.with_suffix(".index")
    dw_file(checkpoint_url, Namer.iter(custom_cache_dir / index_file_name))

if zip_url:
    temp_zip = Path(tempfile.mkstemp(suffix=".zip")[1])
    dw_file(zip_url, temp_zip)
    with zipfile.ZipFile(temp_zip, 'r') as zip_ref:
        zip_ref.extractall(custom_cache_dir)